In [1]:
import pandas as pd
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn import set_config; set_config(display='diagram')

from src.utils import transform_columns, load_data, evaluate, pickle

from xgboost import XGBClassifier

#### Preprocesor pipeline

In [2]:
def preprocess_data(df: pd.DataFrame):
    """
    Prepares the feature preprocessing pipeline.
    """
    # 1. Clean column names: Strip whitespace and replace spaces with underscores
    #df = transform_columns(df)
    # 2. Separate target from features
    # (Assuming the target column name becomes 'churn' after formatting)
    #X = df.drop(columns=['churn', 'customer_id'], errors='ignore')
    X = df.drop(columns=['Churn', 'CustomerID'], errors='ignore')
    y = df['Churn']

    # 3. Identify feature types automatically or explicitly
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

    # 4. Define transformations per data type (Best Practice)
    # Numerical: Impute missing values with median, then scale
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Categorical: Impute missing values with most frequent string, then One-Hot Encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # 5. Bundle transformers into a ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop', # Explicitly drop columns we didn't specify (like IDs)
        verbose_feature_names_out=False
    ).set_output(transform='pandas')

# 6. Train/Validation/Test Split BEFORE fitting anything to avoid data leakage

    # First split: Isolate 20% of the entire dataset for the final Test set
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )

    # Second split: Split the remaining 80% into Train (60%) and Validation (20%)
    # Note: 25% of 80% equals exactly 20% of the original total dataset
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
    )

    # Return all 6 data partitions along with your preprocessor artifact
    return X_train, X_val, X_test, y_train, y_val, y_test, preprocessor

In [3]:
if __name__ == "__main__":
    # Example execution within your Docker container volume path
    data_descriptions_df = load_data('../../raw_data/data_descriptions.csv')
    data_df = load_data('../../raw_data/train.csv')

    print(f"data_descriptions_df shape: {data_descriptions_df.shape}")
    print(f"data_df shape: {data_df.shape}")
    print(f"")

    X_train, X_val, X_test, y_train, y_val, y_test, preprocessor  = preprocess_data(data_df)

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")
    print(f"")
    print(f"X_val shape: {X_val.shape}")
    print(f"y_val shape: {y_val.shape}")

data_descriptions_df shape: (21, 4)
data_df shape: (243787, 21)

X_train shape: (146271, 19)
y_train shape: (146271,)

X_test shape: (48758, 19)
y_test shape: (48758,)

X_val shape: (48758, 19)
y_val shape: (48758,)


In [4]:
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [5]:
data_descriptions_df

,Column_name,Column_type,Data_type,Description
0,AccountAge,Feature,integer,The age of the user's account in months.
1,MonthlyCharges,Feature,float,The amount charged to the user on a monthly ba...
2,TotalCharges,Feature,float,The total charges incurred by the user over th...
3,SubscriptionType,Feature,object,The type of subscription chosen by the user (B...
4,PaymentMethod,Feature,string,The method of payment used by the user.
5,PaperlessBilling,Feature,string,Indicates whether the user has opted for paper...
6,ContentType,Feature,string,The type of content preferred by the user (Mov...
7,MultiDeviceAccess,Feature,string,Indicates whether the user has access to the s...
8,DeviceRegistered,Feature,string,"The type of device registered by the user (TV,..."
9,ViewingHoursPerWeek,Feature,float,The number of hours the user spends watching c...


In [6]:
set_config(display='text')
preprocessor.get_params()

{'force_int_remainder_cols': 'deprecated',
 'n_jobs': None,
 'remainder': 'drop',
 'sparse_threshold': 0.3,
 'transformer_weights': None,
 'transformers': [('num',
   Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                   ('scaler', StandardScaler())]),
   ['AccountAge',
    'MonthlyCharges',
    'TotalCharges',
    'ViewingHoursPerWeek',
    'AverageViewingDuration',
    'ContentDownloadsPerMonth',
    'UserRating',
    'SupportTicketsPerMonth',
    'WatchlistSize']),
  ('cat',
   Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                   ('onehot',
                    OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
   ['SubscriptionType',
    'PaymentMethod',
    'PaperlessBilling',
    'ContentType',
    'MultiDeviceAccess',
    'DeviceRegistered',
    'GenrePreference',
    'Gender',
    'ParentalControl',
    'SubtitlesEnabled'])],
 'verbose': False,
 'verbose_feature_names_out': False,
 'num': Pipeline(steps

In [7]:
X_train.columns

Index(['AccountAge', 'MonthlyCharges', 'TotalCharges', 'SubscriptionType',
       'PaymentMethod', 'PaperlessBilling', 'ContentType', 'MultiDeviceAccess',
       'DeviceRegistered', 'ViewingHoursPerWeek', 'AverageViewingDuration',
       'ContentDownloadsPerMonth', 'GenrePreference', 'UserRating',
       'SupportTicketsPerMonth', 'Gender', 'WatchlistSize', 'ParentalControl',
       'SubtitlesEnabled'],
      dtype='object')

In [8]:
X_train.head(4)

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled
240470,49,15.223830,745.967667,Premium,Bank transfer,No,TV Shows,Yes,Computer,33.477336,30.539721,45,Comedy,1.369308,2,Female,18,Yes,No
121205,116,16.194405,1878.550935,Basic,Credit card,Yes,Movies,Yes,Mobile,33.924589,6.294568,17,Action,2.280899,4,Female,14,Yes,No
50795,59,10.137477,598.111124,Standard,Electronic check,Yes,TV Shows,No,Tablet,32.375993,165.809471,39,Drama,3.205208,7,Male,22,Yes,Yes
138500,102,17.830488,1818.709749,Standard,Mailed check,Yes,TV Shows,Yes,Computer,28.157214,17.397305,11,Comedy,4.382376,2,Female,7,Yes,Yes


In [9]:
X_train.value_counts()

AccountAge  MonthlyCharges  TotalCharges  SubscriptionType  PaymentMethod     PaperlessBilling  ContentType  MultiDeviceAccess  DeviceRegistered  ViewingHoursPerWeek  AverageViewingDuration  ContentDownloadsPerMonth  GenrePreference  UserRating  SupportTicketsPerMonth  Gender  WatchlistSize  ParentalControl  SubtitlesEnabled
1           4.991154        4.991154      Premium           Credit card       No                Both         Yes                Mobile            1.559813             72.489247               20                        Drama            2.860004    7                       Male    4              No               Yes                 1
80          10.703980       856.318431    Premium           Mailed check      No                Movies       Yes                Mobile            3.136553             47.600431               27                        Action           3.381940    9                       Female  13             No               No                  1
         

In [10]:
X_train.describe()

,AccountAge,MonthlyCharges,TotalCharges,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,UserRating,SupportTicketsPerMonth,WatchlistSize
count,146271.000000,146271.000000,146271.000000,146271.000000,146271.000000,146271.000000,146271.000000,146271.000000,146271.000000
mean,60.000677,12.490356,749.921492,20.533364,92.350366,24.514032,3.006567,4.495983,12.016859
std,34.260553,4.331487,522.934469,11.234717,50.499093,14.417059,1.153403,2.872138,7.195162
min,1.000000,4.990112,4.991154,1.000065,5.000937,0.000000,1.000007,0.000000,0.000000
25%,30.000000,8.735979,328.100344,10.823579,48.527618,12.000000,2.008372,2.000000,6.000000
50%,60.000000,12.492649,649.570473,20.592014,92.443263,25.000000,3.006478,4.000000,12.000000
75%,90.000000,16.239561,1087.511126,30.245620,135.981865,37.000000,4.007062,7.000000,18.000000
max,119.000000,19.989957,2378.454499,39.999718,179.999275,49.000000,4.999973,9.000000,24.000000


In [11]:
X_train.isnull().sum()

AccountAge                  0
MonthlyCharges              0
TotalCharges                0
SubscriptionType            0
PaymentMethod               0
PaperlessBilling            0
ContentType                 0
MultiDeviceAccess           0
DeviceRegistered            0
ViewingHoursPerWeek         0
AverageViewingDuration      0
ContentDownloadsPerMonth    0
GenrePreference             0
UserRating                  0
SupportTicketsPerMonth      0
Gender                      0
WatchlistSize               0
ParentalControl             0
SubtitlesEnabled            0
dtype: int64

In [12]:
def train_xgboost(X_train, y_train, X_test, y_test, preprocessor):
    """
    Assembles the preprocessing and XGBoost steps into a single pipeline,
    trains the model, and evaluates performance.
    """
    model = XGBClassifier(
            n_estimators=200,           # Number of gradient boosted trees to build
            max_depth=5,                # Maximum depth of each tree (prevents overfitting)
            learning_rate=0.05,         # Step size shrinkage to make training stable
            objective='binary:logistic',# Mandates a probability output between 0 and 1
            eval_metric='auc',          # Optimizes for your 18% class imbalance
            random_state=42,            # Ensures reproducibility
            n_jobs=-1                   # Uses all available CPU cores in your Docker setup
        )
    # 1. Create a unified pipeline
    # The preprocessor output (Pandas DataFrame) automatically feeds straight into XGBoost
    #pipeline = Pipeline(steps=[
    #    ('preprocessor', preprocessor),
    #    ('classifier', model)
    #])

    # 2. Train (Fit) the entire pipeline
    # This automatically fits the transformers, transforms the data, and trains XGBoost
    #print("Training XGBoost pipeline...")
    preprocessor.fit(X_train, y_train)
    X_train = preprocessor.transform(X_train)
    X_test = preprocessor.transform(X_test)

    model.fit(X_train, y_train)

    evaluate(preprocessor, model, X_test, y_test)

    # 4. Serialize and export the pipeline artifact to your Shared Volume
    pickle(preprocessor, model)

In [13]:
train_xgboost(X_train, y_train, X_test, y_test, preprocessor)


================ MODEL PERFORMANCE ================
              precision    recall  f1-score   support

           0       0.83      0.98      0.90     39921
           1       0.58      0.11      0.18      8837

    accuracy                           0.82     48758
   macro avg       0.70      0.54      0.54     48758
weighted avg       0.79      0.82      0.77     48758

ROC-AUC Score: 0.7506
============================================= =======
Success! Saved pipeline preprocessing artifact to: /Users/annamorgiel/code/soodabeh/retention-agent/notebooks/models/xgb_churn_preproc_pipeline.pkl
Success! Saved model artifact to: /Users/annamorgiel/code/soodabeh/retention-agent/notebooks/models/xgb_churn_model.pkl


In [14]:
preprocessor.get_params()

{'force_int_remainder_cols': 'deprecated',
 'n_jobs': None,
 'remainder': 'drop',
 'sparse_threshold': 0.3,
 'transformer_weights': None,
 'transformers': [('num',
   Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                   ('scaler', StandardScaler())]),
   ['AccountAge',
    'MonthlyCharges',
    'TotalCharges',
    'ViewingHoursPerWeek',
    'AverageViewingDuration',
    'ContentDownloadsPerMonth',
    'UserRating',
    'SupportTicketsPerMonth',
    'WatchlistSize']),
  ('cat',
   Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                   ('onehot',
                    OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
   ['SubscriptionType',
    'PaymentMethod',
    'PaperlessBilling',
    'ContentType',
    'MultiDeviceAccess',
    'DeviceRegistered',
    'GenrePreference',
    'Gender',
    'ParentalControl',
    'SubtitlesEnabled'])],
 'verbose': False,
 'verbose_feature_names_out': False,
 'num': Pipeline(steps

In [15]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['AccountAge', 'MonthlyCharges',
                                  'TotalCharges', 'ViewingHoursPerWeek',
                                  'AverageViewingDuration',
                                  'ContentDownloadsPerMonth', 'UserRating',
                                  'SupportTicketsPerMonth', 'WatchlistSize']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                     

#### Save unprocessed validation set

In [16]:
#Validation data
X_val_transformed = preprocessor.transform(X_val)
X_val = pd.DataFrame(X_val_transformed)
y_val = pd.DataFrame(y_val)

In [17]:
BASE_PATH = '../../data/'
os.makedirs(BASE_PATH, exist_ok=True)
file_path_X = os.path.join(BASE_PATH, 'X_val_xgb.csv')
file_path_y = os.path.join(BASE_PATH, 'y_val_xgb.csv')
# 3. Save the file (and remove the trailing comma)
# CRITICAL: Use index=False so Pandas doesn't inject an extra row-number column
X_val.to_csv(file_path_X, index=False)
y_val.to_csv(file_path_y, index=False)